# beat2body: team notebook

Runs the beat2body evaluations and analysis on Colab. Full guide: [docs/TEAM.md](https://github.com/novaz03/beat2body/blob/claude/relaxed-knuth-fc6tqf/docs/TEAM.md).

**Once:**
1. Runtime → Change runtime type → **GPU**. Only Task A needs it.
2. Key icon on the left → add a secret **`HF_TOKEN`**: your Hugging Face token with write access to `SizheZ03/beat2body-runs`. Turn on notebook access.
3. **Never type a token into a cell.** This notebook reads it from the secret.

Then run the **Setup** cell and the task you picked from the table at the bottom of TEAM.md.

## Setup

In [ ]:
WORKER = ""                                   # <- your name, e.g. "alice": shown on the evaluations you run
BRANCH = "claude/relaxed-knuth-fc6tqf"

import os, subprocess
from google.colab import userdata
assert WORKER, "put your name in WORKER first"
os.environ["BEAT2BODY_WORKER"] = WORKER
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")    # from the Colab secret, never typed here
print("ready:", WORKER)

## Task A: run the evaluation queue (GPU)

Fetches the **latest** `scripts/colab_eval.py` from GitHub and runs it:
- it claims each queued evaluation before running it, so teammates running this at the same time skip it;
- it uploads results to Hugging Face;
- then it makes any queued videos.

Takes about 6–8 minutes per evaluation. Run it again whenever new work is queued; finished entries are skipped.

In [ ]:
import urllib.request
src = urllib.request.urlopen(
    f"https://raw.githubusercontent.com/novaz03/beat2body/{BRANCH}/scripts/colab_eval.py").read().decode()
exec(compile(src, "colab_eval.py", "exec"), {"__name__": "__main__"})

## Task B: per-genre analysis (no GPU)

Collects the per-genre results from Hugging Face into tables:
- a **genre difficulty** ranking: each model's FID divided by its genre's real reference, 1 = as close as real dance;
- one table per genre, with every model next to the real values.

Saved as `per_genre.md`; download it from the Files panel on the left.

Do not compare raw FIDs across genres. See TEAM.md, Task B.

In [ ]:
REPO = "/content/beat2body"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, "--filter=blob:none", "--sparse",
                    "https://github.com/novaz03/beat2body", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q", "--ff-only"], check=True)
subprocess.run(["pip", "-q", "install", "huggingface_hub"], check=True)
subprocess.run(["python", f"{REPO}/scripts/summarize_genres.py", "--out", "/content/per_genre.md"], check=True)
from IPython.display import Markdown, display
display(Markdown(open("/content/per_genre.md").read()))

## Task C: video review (no GPU)

Each video shows the real dancer, then 6.1, 6.2 and 6.3, for 20 s with music. The panels flash yellow on beats.

Set `GENRE` and run the cell to watch a genre: `ballet_jazz`, `break`, `house`, `krump`, `la_hiphop`, `lock`, `middle_hiphop`, `pop`, `street_jazz`, `waack`.

Note, with timestamps:
- feet sliding;
- body collapse or tangles;
- shrinking or turning away from the camera;
- big moves on the beat flashes;
- whether it fits the genre.

The notes go in `reports/2026-11-24_genres/REVIEW.md` (see TEAM.md, Task C).

In [ ]:
GENRE = "ballet_jazz"

from huggingface_hub import snapshot_download
from IPython.display import Video, display
import glob
d = snapshot_download("SizheZ03/beat2body-runs", allow_patterns=["reports/2026-11-23_v6/videos/*"],
                      token=os.environ["HF_TOKEN"], local_dir="/content/review")
for f in sorted(glob.glob(f"{d}/reports/2026-11-23_v6/videos/*_{GENRE}_*.mp4")):
    print(os.path.basename(f))
    display(Video(f, embed=True, width=960))